### Require variable

In [2]:
import sys
import os




project_path = r"C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder"
if project_path not in sys.path:
    sys.path.append(project_path)

import track_builder as tb

import pandas as pd
import track_builder as tb

import os
from dotenv import load_dotenv

load_dotenv()

saving_path = os.getenv("SAVING")
track_output_path = os.path.join(saving_path, "tracks")
df_output_path = os.path.join(saving_path, "df")
os.makedirs(track_output_path, exist_ok=True)
os.makedirs(df_output_path, exist_ok=True)

BASE_PATH = r"C:\Users\lamin\Documents\maitrise\ASTD\data"
YEAR      = 2019

MONTHS_TO_LOAD = [1, 2, 3, 4, 5]

USECOLS   = "default"
SAMPLING  = [0, -1]

COLS_REQUIRED = [
    "shipid",
    "date_time_utc",
    "latitude",
    "longitude",
    "astd_cat",
    "flagname",
    "iceclass",
    "sizegroup_gt"
]


### Loading the first and last day of 5 month of data

In [3]:
df_parquet_file = os.path.join(df_output_path, "first_and_last_of_5month_astd_data.parquet")

if os.path.exists(df_parquet_file):
    df_for_track = pd.read_parquet(df_parquet_file)
else:
    df_for_track = tb.load_astd_monthly(
        BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
        usecols=USECOLS, sampling=SAMPLING, remove_nan_rows=COLS_REQUIRED
    )

    
    df_for_track.to_parquet(df_parquet_file, index=False)



### Load all data for 5 month

In [4]:
df_parquet_file2 = os.path.join(df_output_path, "all5month_astd_data2_without_nan_col.parquet")

if os.path.exists(df_parquet_file2):
    df = pd.read_parquet(df_parquet_file2)
else:
    df = tb.load_astd_monthly(
        BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
        usecols=USECOLS, sampling=None, remove_nan_rows=COLS_REQUIRED
    )
    
    df.to_parquet(df_parquet_file2, index=False)


### Build the track table

In [5]:
parquet_file_path = os.path.join(track_output_path, "tracks_2019_for_5_month.parquet")

if os.path.exists(parquet_file_path):
    tracks = pd.read_parquet(parquet_file_path)
else:
    tracks = tb.build_ship_tracks(df_for_track,
                                  max_time_gap_hours=25,
                                  max_distance_km=400,
                                  min_track_length=1,
                                  matching_strategy="balanced",
                                  )
    tracks.to_parquet(parquet_file_path)

tracks

,month,segment_id,track_id
0,2019-01,5966,1
1,2019-01,3768,2
2,2019-01,3994,3
3,2019-01,1892,4
4,2019-01,3009,5
...,...,...,...
3990,2019-05,7601,2913
3991,2019-05,15938,2914
3992,2019-05,4535,2915
3993,2019-05,9630,2916


### Load 10 track across 5 month to visualize

In [6]:

work = tb.build_light_multi_track_data(track_table=tracks, track_sampling=10, positions_df=df, n_tracks_length=5, preprocess_positions=True)

fig = tb.plot_ship_tracks(
    work,
    color_by="track_id",
    color_mode="categorical",
    show_points=False,
    map_style="open-street-map",
    title="Tracks (light multi-track sample)",
)
fig.update_layout(showlegend=False)
fig.show()

computing typical speeds...


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:358: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')
C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:166: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_bwd = valid_fwd.shift(1).fillna(False) & is_same_ship_prev


Cleaning completed: 33452 'ghost' or aberrant points removed.


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\io\astd_loader.py:501: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.iloc[::point_stride])


### Load data for one specific track

In [7]:
track_904 = tb.load_track_data(track_ids=904, track_table=tracks, base_path=BASE_PATH, chunksize=500_000)

c:\Users\lamin\miniconda3\envs\torch-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

Batch loading 1 tracks: 100%|██████████| 5/5 [02:21<00:00, 28.27s/it]


computing typical speeds...
Cleaning completed: 3429 'ghost' or aberrant points removed.


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:358: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:166: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



### plot the track 904

In [8]:
fig_track_904 = tb.plot_individual_track(
        track_id=904,
        track_table=tracks,
        astd_data=track_904,
        show_segments=True,
        map_style="open-street-map",
        title=f"Track {904}",
    )
    
fig_track_904.show()

### You can use also the function `build_light_multi_track_data` to plot one specific track

In [9]:
work = tb.build_light_multi_track_data(track_table=tracks, specific_track_ids=[904], positions_df=df)

fig = tb.plot_ship_tracks(
    work,
    color_by="track_id",
    color_mode="categorical",
    show_points=False,
    map_style="open-street-map",
    title="Tracks (light multi-track sample)",
)
fig.update_layout(showlegend=False)
fig.show()

computing typical speeds...


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:358: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:166: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\io\astd_loader.py:501: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups

Cleaning completed: 3429 'ghost' or aberrant points removed.
